In [0]:
%run "/Users/gresahasani19@gmail.com/urban-mobility-lakehouse/src/monitoring/pipeline_metrics"

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

_started = datetime.now()

trips = spark.table("urban_mobility.silver.trips_enriched").filter(F.col("trip_status") == "COMPLETED")
_records_read = trips.count()

pct = trips.approxQuantile(
    ["average_speed_kmh", "total_amount", "trip_duration_minutes", "revenue_per_km"],
    [0.995],
    0.001
)

SPEED_THRESHOLD = max(pct[0][0], 100)
FARE_THRESHOLD = max(pct[1][0], 150)
DURATION_THRESHOLD = max(pct[2][0], 120)
RPK_THRESHOLD = max(pct[3][0], 50)

print(f"Thresholds -> speed: {SPEED_THRESHOLD:.2f} km/h | fare: {FARE_THRESHOLD:.2f} | duration: {DURATION_THRESHOLD:.2f} min | revenue/km: {RPK_THRESHOLD:.2f}")

anomalies = (
    trips
    .withColumn(
        "anomaly_type",
        F.when(F.col("average_speed_kmh") > SPEED_THRESHOLD, "IMPOSSIBLE_SPEED")
         .when(F.col("total_amount") > FARE_THRESHOLD, "EXTREME_FARE")
         .when(F.col("trip_duration_minutes") > DURATION_THRESHOLD, "EXTREME_DURATION")
         .when(F.col("revenue_per_km") > RPK_THRESHOLD, "DISTANCE_FARE_MISMATCH")
         .otherwise(None)
    )
    .filter(F.col("anomaly_type").isNotNull())
    .select(
        "trip_id", "driver_id", "pickup_zone_id", "dropoff_zone_id",
        "distance_km", "trip_duration_minutes", "average_speed_kmh",
        "fare_amount", "total_amount", "revenue_per_km",
        "anomaly_type", "pickup_datetime"
    )
    .withColumn("detected_at", F.current_timestamp())
)

anomalies.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.gold.trip_anomalies")
_records_written = spark.table("urban_mobility.gold.trip_anomalies").count()
_completed = datetime.now()

log_pipeline_run(
    pipeline_name="gold_layer",
    task_name="trip_anomalies",
    started_at=_started,
    completed_at=_completed,
    records_read=_records_read,
    records_written=_records_written,
    records_rejected=0,
    status="SUCCESS"
)

print("gold.trip_anomalies rows:", _records_written)
spark.table("urban_mobility.gold.trip_anomalies").groupBy("anomaly_type").count().show()